# Notebook 2: Nghiên Cứu & Cài Đặt Thuật Toán Tách Sọ Não (Skull Stripping)
**Đồ án Xử lý ảnh số (2026–2027) — Khoa CNTT — HUIT**  
**Nhóm sinh viên:** Hà Mạnh Trường (2045240326) & Huỳnh Tâm Trí (2045240315)  
**Phụ trách chính:** Hà Mạnh Trường

---
### Cơ sở lý thuyết DIP:
1. **Khử nhiễu Rician:** Lọc Bilateral (`cv2.bilateralFilter`) bảo toàn biên đối tượng.
2. **Nhị phân hóa:** Ngưỡng tự động Otsu (`cv2.THRESH_OTSU`).
3. **Hình thái học:** Phép mở (`MORPH_OPEN`) với phần tử cấu trúc elip cắt đứt cầu nối mô.
4. **Trích xuất nhu mô:** Tìm thành phần liên thông lớn nhất (Largest Connected Component - LCC).
5. **Lấp đầy lỗ hổng:** Kỹ thuật FloodFill lấp đầy não thất.


In [ ]:
import os
import sys
import numpy as np
import cv2
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from src.preprocessing.filters import apply_bilateral_filter
from src.preprocessing.skull_strip import extract_brain_mask, strip_skull


### 1. Phân Tích Chi Tiết Từng Bước Tách Sọ Trên 1 Ca Bệnh


In [ ]:
# Nạp ảnh ca bệnh 01
img_path = "../data/sample/images/mri_brain_tumor_01.png"
img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

# Bước 1: Khử nhiễu Bilateral
denoised = apply_bilateral_filter(img, d=9, sigma_color=75.0, sigma_space=75.0)

# Bước 2: Ngưỡng hóa Otsu
_, otsu_thresh = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Bước 3: Phép mở hình thái học (Morphological Opening)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
opened = cv2.morphologyEx(otsu_thresh, cv2.MORPH_OPEN, kernel, iterations=2)

# Bước 4: Trích xuất thành phần liên thông lớn nhất (LCC)
num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(opened, connectivity=8)
largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
lcc_mask = np.zeros_like(img)
lcc_mask[labels == largest_label] = 255

# Bước 5: Lấp đầy lỗ hổng (Hole Filling)
filled_mask = extract_brain_mask(denoised, kernel_size=5)

# Bước 6: Ảnh não sạch sọ
stripped, _ = strip_skull(denoised, filled_mask)

# Vẽ biểu đồ 6 bước
steps = [
    ("1. Ảnh MRI Gốc", img, 'gray'),
    ("2. Ngưỡng Hóa Otsu", otsu_thresh, 'gray'),
    ("3. Morphological Opening", opened, 'gray'),
    ("4. LCC Nhu Mô Não", lcc_mask, 'gray'),
    ("5. Brain Mask (Filled)", filled_mask, 'gray'),
    ("6. Não Sau Tách Sọ", stripped, 'gray')
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for i, (title, step_img, cmap) in enumerate(steps):
    r, c = i // 3, i % 3
    axes[r, c].imshow(step_img, cmap=cmap)
    axes[r, c].set_title(title, fontsize=12, fontweight='bold')
    axes[r, c].axis('off')

plt.suptitle("Quy Trình 6 Bước Tách Sọ Não Bằng Kỹ Thuật Xử Lý Ảnh Số", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 2. Kiểm Thử Tính Bền Vững Trên 4 Ca Khác Nhau


In [ ]:
test_cases = ["mri_brain_tumor_02", "mri_brain_tumor_07", "mri_brain_tumor_11", "mri_brain_tumor_17"]

fig, axes = plt.subplots(4, 3, figsize=(12, 14))

for idx, case_name in enumerate(test_cases):
    p = f"../data/sample/images/{case_name}.png"
    im = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    mask = extract_brain_mask(im)
    res, _ = strip_skull(im, mask)
    
    axes[idx, 0].imshow(im, cmap='gray')
    axes[idx, 0].set_title(f"Gốc: {case_name}")
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(mask, cmap='hot')
    axes[idx, 1].set_title("Brain Mask")
    axes[idx, 1].axis('off')
    
    axes[idx, 2].imshow(res, cmap='gray')
    axes[idx, 2].set_title("Não Sạch Sọ")
    axes[idx, 2].axis('off')

plt.tight_layout()
plt.show()
